**Relevant physical groups**
1. Per-hit charge info
    * cluster_Charge
    * cluster_DeDxStrip
    * cluster_SaturatingStrips
    * cluster_DetId
2. Track geometry/kinematics
    * IsoTrack_eta, IsoTrack_p, IsoTrack_pt, IsoTrack_phi, IsoTrack_dxy, IsoTrack_dz, IsoTrack_numberOfValidHits, IsoTrack_numberOfTrackerLayers, IsoTrack_numberOfValidPixelHits
3. Track-quality flags
    * IsoTrack_isHighPurityTrack, IsoTrack_normChi2, IsoTrack_fractionOfValidHits

4. Event metadata
    * run, luminosityBlock, event, isData, HSCP_hasDeDx

5. Truth (for MC validation)
    * GenPart_pdgId, GenPart_eta, GenPart_p, GenPart_beta, GenPart_mass, GenPart_charge

In [1]:
!python ../../src/utils.py
%reload_ext autoreload

In [ ]:
import awkward as ak
import hist
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from coffea import processor

NanoAODSchema.warn_missing_crossrefs = False

In [ ]:
# %pip install -e ../..     # two levels up from src/adaptive_trunc to repo root


Obtaining file:///home/bothsides/projects/optimizing_DEDx_estimator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for adaptive-trunc (pyproject.toml) ... done
  Created wheel for adaptive-trunc: filename=adaptive_trunc-0.0.1-0.editable-py3-none-any.whl size=1288 sha256=b76a5d1b7e1f1f15ef1dc723fa9f152c282adba349c1efb3f6b2e6008981f23f
  Stored in directory: /tmp/pip-ephem-wheel-cache-1c5mvnrg/wheels/24/94/90/8b74b357a944ee46bd2edeaa71ff197a8e4ff1ca15f1b6357e
Successfully built adaptive-trunc
Note: you may need to restart the kernel to use updated packages.


In [32]:
import os
import math
import ROOT as rt
import numpy as np
import matplotlib.pyplot as plt
import copy
import utils 
from ROOT import VecOps
from analysis import (df1, tree1, HMNCSBR, TRUNCSBR, COLOR_MAP)
name_tag = str(utils.TAG)
print("cwd:", os.getcwd())



cwd: /home/bothsides/projects/optimizing_DEDx_estimator/src/adaptive_trunc


In [31]:
print(f"number of events {df1.Count().GetValue()}")

for branch in tree1.GetListOfBranches():
    for leaf in branch.GetListOfLeaves():
        print(f"Branch `{branch.GetName()}` → Leaf `{leaf.GetName()}` : {leaf.GetTypeName()}")

number of events 27162
Branch `DeDx_FiPixel` → Leaf `DeDx_FiPixel` : vector<float>
Branch `DeDx_FiPixelNoL1` → Leaf `DeDx_FiPixelNoL1` : vector<float>
Branch `DeDx_Gi` → Leaf `DeDx_Gi` : vector<float>
Branch `DeDx_GiStrip` → Leaf `DeDx_GiStrip` : vector<float>
Branch `DeDx_Ih` → Leaf `DeDx_Ih` : vector<float>
Branch `DeDx_IhNOM` → Leaf `DeDx_IhNOM` : vector<unsigned int>
Branch `DeDx_IhNoL1` → Leaf `DeDx_IhNoL1` : vector<float>
Branch `DeDx_IhNoL1NOM` → Leaf `DeDx_IhNoL1NOM` : vector<unsigned int>
Branch `DeDx_IhPixel` → Leaf `DeDx_IhPixel` : vector<float>
Branch `DeDx_IhPixelNoL1` → Leaf `DeDx_IhPixelNoL1` : vector<float>
Branch `DeDx_IhStrip` → Leaf `DeDx_IhStrip` : vector<float>
Branch `DeDx_IhStrip1` → Leaf `DeDx_IhStrip1` : vector<float>
Branch `DeDx_IhStrip3` → Leaf `DeDx_IhStrip3` : vector<float>
Branch `DeDx_IhStrip4` → Leaf `DeDx_IhStrip4` : vector<float>
Branch `DeDx_It` → Leaf `DeDx_It` : vector<float>
Branch `DeDx_ItStrip` → Leaf `DeDx_ItStrip` : vector<float>
Branch `DeDx_

In [13]:
a = []
for branch in tree1.GetListOfBranches():
    if branch.GetName() == "cluster_DeDxStrip":
        a = branch
        break

In [29]:
cluster = df1.AsNumpy(["cluster_DeDxStrip"])["cluster_DeDxStrip"]
len(cluster)
for stuff in cluster:
    if len(stuff) > 5:
        print(stuff)
        

{ { 2.90608f, 3.82821f, 3.12415f, 2.83457f, 2.67552f, 2.39453f, 3.35673f, 3.44683f }, { 3.61907f, 7.57861f, 2.63913f, 3.13592f, 2.98862f, 2.48933f, 2.75801f, 4.06988f, 4.24154f, 4.87823f, 2.54436f, 3.08469f }, {}, {}, { 3.67237f, 3.05512f, 4.28230f, 3.49360f, 2.59189f, 3.82056f, 4.03441f, 4.32050f, 3.35954f, 3.89873f, 2.99745f }, { 2.93108f, 2.58718f, 3.63254f, 3.47732f, 2.89702f, 4.75840f, 3.25425f, 3.75348f, 4.83529f, 2.75261f, 2.17558f } }
{ { 3.00330f, 3.34526f, 6.79932f, 3.09282f, 3.30674f, 3.18440f, 2.78319f, 2.91799f, 2.15581f, 3.37807f, 3.60862f, 3.33915f, 2.99512f, 2.88201f }, { 3.61795f, 5.52271f, 3.73825f, 7.25302f, 2.53164f, 1.94109f, 3.50323f, 3.06332f, 1.84743f, 7.49097f, 4.72935f, 4.26685f, 4.28635f, 4.62786f, 4.06007f, 2.93569f, 2.34360f }, { 2.71657f, 13.1014f, 3.17649f, 3.17465f, 3.20224f, 5.44354f, 7.81769f, 5.03980f, 5.31907f, 3.02514f }, { 3.90282f, 2.73117f, 3.83984f, 2.85366f, 3.03038f, 3.27667f, 3.18274f, 4.70408f, 3.75661f, 3.35194f, 2.42525f, 4.50310f, 2.96288

In [ ]:
# ---------------------------
# Gaussianity trimmer
# ---------------------------
def gaussianity_score(x):
    """
    Return skewness and excess kurtosis absolute-weighted score.
    x: 1D numpy array
    """
    n = x.size
    if n < 3:
        return 1e9
    m = x.mean()
    v = x.var(ddof=1)
    if v <= 0:
        return 1e9
    g1 = ((x - m) ** 3).mean() / (v ** 1.5)
    g2 = ((x - m) ** 4).mean() / (v ** 2) - 3.0
    J = abs(g1) + 0.3 * abs(g2)
    return J


27162
